# Panel series diagnostics

This notebook demonstrates independent panel diagnostics and how to combine their summaries with DataFrame merges.

In [1]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyshift.series import (
    IntermittencyAnalyzer,
    PAMIAnalyzer,
    PredictabilityAnalyzer,
    SeasonalityAnalyzer,
    TrendAnalyzer,
    VarianceRatioAnalyzer,
)

In [2]:
n = 84
steps = np.arange(n)
dates = pd.date_range("2025-01-01", periods=n, freq="D")
rng = np.random.default_rng(42)

smooth = 6 + 1.5 * np.sin(2 * np.pi * steps / 7) + 0.02 * steps
intermittent = np.where(steps % 3 == 0, 3 + rng.uniform(0, 2, n), 0.0)
lumpy = np.where(steps % 4 == 0, rng.lognormal(1.2, 0.9, n), 0.0)

df = pd.concat(
    [
        pd.DataFrame({"unique_id": uid, "ds": dates, "y": values})
        for uid, values in [
            ("smooth_weekly", smooth),
            ("intermittent", intermittent),
            ("lumpy", lumpy),
        ]
    ],
    ignore_index=True,
)
df.head()

,unique_id,ds,y
0,smooth_weekly,2025-01-01,6.000000
1,smooth_weekly,2025-01-02,7.192747
2,smooth_weekly,2025-01-03,7.502392
3,smooth_weekly,2025-01-04,6.710826
4,smooth_weekly,2025-01-05,5.429174


## Demand occurrence

`IntermittencyAnalyzer.summary()` returns the occurrence and magnitude diagnostics for each series. Detailed interval arrays remain available in `results_`.

In [3]:
intermittency = IntermittencyAnalyzer().fit(df)
intermittency.summary()

,unique_id,adi,cv2,zero_proportion,interval_cv,classification
0,intermittent,3.0,0.013614,0.666667,0.0,intermittent
1,lumpy,4.0,1.253091,0.750000,0.0,lumpy
2,smooth_weekly,1.0,0.027822,0.000000,0.0,smooth


## Candidate seasonal periods

The detector profile exposes only `candidate_periods`; FFT frequencies, power, and peaks can be inspected through `results_`.

In [4]:
seasonality = SeasonalityAnalyzer(top_k=2).fit(df)
seasonality.summary()

,unique_id,candidate_periods,significant_periods
0,intermittent,"[3, 6]",[3]
1,lumpy,"[2, 4]","[2, 4]"
2,smooth_weekly,[7],[7]


## Variance ratio by horizon

`VarianceRatioAnalyzer` returns one diagnostic per SKU and horizon. Ratios above 1 indicate persistent changes; ratios below 1 indicate mean-reverting changes.

In [5]:
variance_ratios = VarianceRatioAnalyzer().fit(df)
variance_ratios.summary()

,unique_id,horizon,variance_ratio,z_statistic,p_value
0,intermittent,2,0.512476,-4.441559,8.930949e-06
1,intermittent,4,0.260014,-3.603533,3.139203e-04
2,intermittent,8,0.139486,-2.650287,8.042339e-03
3,lumpy,2,0.512358,-4.442634,8.886413e-06
4,lumpy,4,0.155191,-4.113993,3.888733e-05
5,lumpy,8,0.068353,-2.869368,4.112936e-03
6,smooth_weekly,2,1.639561,5.826682,5.654025e-09
7,smooth_weekly,4,1.324506,1.580258,1.140478e-01
8,smooth_weekly,8,0.138114,-2.654513,7.942301e-03


## PAMI local minima and forecasting lags

`PAMIAnalyzer` keeps lag discovery separate from lag creation. Its summary contains only each SKU and the local minima found in its PAMI curve. Fallback behavior is applied later, when model-compatible lags are created.

In [6]:
pami = PAMIAnalyzer(max_tau=30, normalize=True).fit(df)
pami.summary()

,unique_id,local_minima
0,intermittent,"[4, 7, 10, 13, 18, 22, 25, 27]"
1,lumpy,"[2, 6, 10, 14, 18, 22, 26]"
2,smooth_weekly,"[2, 5, 9, 12, 16, 19, 23, 26]"


In [7]:
# Dictionary accepted directly by DTLWrapper or DMSTLWrapper as `nlags`.
pami.lags(mode="short", short=2, fallback=1)

{'intermittent': [1, 2, 4], 'lumpy': [1, 2], 'smooth_weekly': [1, 2]}

## Predictability and trend

These analyzers expose complementary diagnostic sections for the same panel.

In [13]:
PredictabilityAnalyzer(detrend="linear", permutation_m=5).fit(df).summary()

,unique_id,foreca,limit,spectral_concentration
0,intermittent,0.954999,0.674687,0.957805
1,lumpy,0.197007,0.674238,0.074067
2,smooth_weekly,0.992008,0.593738,0.992513


In [9]:
TrendAnalyzer(significance_level=0.05).fit(df).summary()

,unique_id,trend_slope,trend_r2,trend_pvalue,significant_trend
0,intermittent,-0.002878,0.001274,0.747201,False
1,lumpy,0.014700,0.008430,0.406184,False
2,smooth_weekly,0.017351,0.136375,0.000546,True


## Combining analyzer summaries

Each compact summary has one row per SKU, so the outputs can be combined explicitly with validated one-to-one merges.

In [10]:
analyzers = [
    IntermittencyAnalyzer(),
    PredictabilityAnalyzer(),
    TrendAnalyzer(),
    SeasonalityAnalyzer(top_k=2),
]
summaries = [analyzer.fit(df).summary() for analyzer in analyzers]
combined = summaries[0]
for summary in summaries[1:]:
    combined = combined.merge(summary, on="unique_id", validate="one_to_one")
combined

,unique_id,adi,cv2,zero_prop,interval_cv,class,foreca,limit,spectral_concentration,trend_slope,trend_r2,trend_pvalue,significant_trend,candidate_periods,significant_periods
0,intermittent,3.0,0.013614,0.666667,0.0,intermittent,0.954999,0.386935,0.957805,-0.002878,0.001274,0.747201,False,"[3, 6]",[3]
1,lumpy,4.0,1.253091,0.750000,0.0,lumpy,0.197007,0.419804,0.074067,0.014700,0.008430,0.406184,False,"[2, 4]","[2, 4]"
2,smooth_weekly,1.0,0.027822,0.000000,0.0,smooth,0.992008,0.281646,0.992513,0.017351,0.136375,0.000546,True,[7],[7]
